# Clase 11 — Práctica: casos de uso reales

En la clase de QAOA resolviste Max-Cut sobre un grafo ciclo de seis nodos. En esta práctica cierras el curso aplicando el mismo algoritmo a un grafo distinto y reflexionas sobre una aplicación real de lo que aprendiste.

## Parte A — QAOA sobre un grafo distinto

Antes de empezar, instala las librerías necesarias si trabajas en Colab:

`!pip install -q cirq tensorflow tensorflow-quantum networkx`

### Enunciado

En el capítulo de QAOA resolviste Max-Cut sobre un grafo ciclo de seis nodos (`networkx.cycle_graph(6)`). Ahora aplicas el mismo circuito QAOA a un grafo completo de cinco nodos (`networkx.complete_graph(5)`), donde cada nodo está conectado con todos los demás. Adapta el andamiaje de código, entrena el modelo y compara el corte medio obtenido con el máximo teórico calculado por fuerza bruta.

**Pistas**

- Reutiliza el hamiltoniano de mezcla y el hamiltoniano de coste tal como se construyen en el capítulo de QAOA: solo cambia el grafo de entrada
- Con cinco nodos, la fuerza bruta sigue siendo rápida: puedes calcular el corte máximo real para comparar
- Deja `p = 1` como en el ejemplo original, así el circuito y el entrenamiento tardan lo mismo
- Los símbolos entrenables siguen siendo `gamma` y `beta`, uno para el bloque de coste y otro para el de mezcla

In [ ]:
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import itertools


def construir_hamiltoniano_mezcla(qubits):
    """Hamiltoniano de mezcla: igual que en el capítulo de QAOA, una suma de X sobre cada qubit."""
    return sum(cirq.PauliString(cirq.X(qubit)) for qubit in qubits)


def construir_hamiltoniano_coste(grafo, qubits):
    """Hamiltoniano de coste: igual que en el capítulo de QAOA, un término Z_i·Z_j por cada arista."""
    hamiltoniano = grafo.number_of_edges() / 2
    for nodo_a, nodo_b in grafo.edges():
        hamiltoniano += cirq.PauliString(0.5 * cirq.Z(qubits[nodo_a]) * cirq.Z(qubits[nodo_b]))
    return hamiltoniano


# TODO: crea el grafo completo de cinco nodos con networkx.complete_graph(5) y guárdalo en 'grafo'
grafo = None

qubits = [cirq.GridQubit(0, i) for i in range(grafo.number_of_nodes())]

estado_inicial = cirq.Circuit()
for qubit in qubits:
    estado_inicial.append(cirq.H(qubit))  # superposición de todas las divisiones posibles

hamiltoniano_mezcla = construir_hamiltoniano_mezcla(qubits)
hamiltoniano_coste = construir_hamiltoniano_coste(grafo, qubits)

p = 1  # número de repeticiones del ciclo coste-mezcla
gamma, beta = sympy.symbols("gamma beta")
circuito_qaoa = cirq.Circuit()
circuito_qaoa += tfq.util.exponential(operators=[hamiltoniano_coste, hamiltoniano_mezcla],
                                       coefficients=[gamma, beta])

entradas = tfq.convert_to_tensor([estado_inicial])
capa_entrada = tf.keras.layers.Input(shape=(), dtype=tf.dtypes.string)
salida = tfq.layers.PQC(circuito_qaoa, hamiltoniano_coste)(capa_entrada)
modelo_qaoa = tf.keras.models.Model(inputs=capa_entrada, outputs=salida)
modelo_qaoa.compile(loss=tf.keras.losses.MAE, optimizer=tf.keras.optimizers.Adam())

In [ ]:
# TODO: entrena modelo_qaoa con .fit(entradas, np.array([0]), epochs=200) y guarda el resultado en 'historial'
historial = None

plt.plot(historial.history["loss"])
plt.title("QAOA sobre un grafo completo de 5 nodos — pérdida durante el entrenamiento")
plt.xlabel("Iteración")
plt.ylabel("Pérdida")
plt.show()

# Muestreamos el circuito entrenado para obtener divisiones concretas
parametros_entrenados = modelo_qaoa.trainable_variables
circuito_con_datos = tfq.layers.AddCircuit()(entradas, append=circuito_qaoa)

# TODO: muestrea el circuito con tfq.layers.Sample(), pasando circuito_con_datos, symbol_names=[gamma, beta],
# symbol_values=parametros_entrenados y repetitions=1000; guarda el resultado en 'muestras'
muestras = None

cortes_obtenidos = []
for bits in muestras.values:
    grupo_uno = [i for i, bit in enumerate(bits) if bit == 1]
    cortes_obtenidos.append(nx.algorithms.cuts.cut_size(grafo, grupo_uno))

# TODO: calcula el corte máximo real por fuerza bruta (igual que en el capítulo de QAOA), probando
# todos los subconjuntos posibles de nodos con itertools.combinations(grafo.nodes(), r) para cada
# tamaño r, y guárdalo en 'mejor_corte'
mejor_corte = None

print("Corte medio obtenido:", np.mean(cortes_obtenidos))
print("Mejor corte posible (fuerza bruta):", mejor_corte)

<details>
<summary>💡 Ver solución</summary>

```python
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import itertools


def construir_hamiltoniano_mezcla(qubits):
    """Hamiltoniano de mezcla: igual que en el capítulo de QAOA, una suma de X sobre cada qubit."""
    return sum(cirq.PauliString(cirq.X(qubit)) for qubit in qubits)


def construir_hamiltoniano_coste(grafo, qubits):
    """Hamiltoniano de coste: igual que en el capítulo de QAOA, un término Z_i·Z_j por cada arista."""
    hamiltoniano = grafo.number_of_edges() / 2
    for nodo_a, nodo_b in grafo.edges():
        hamiltoniano += cirq.PauliString(0.5 * cirq.Z(qubits[nodo_a]) * cirq.Z(qubits[nodo_b]))
    return hamiltoniano


# Cambio clave: un grafo completo de 5 nodos en vez del ciclo de 6 nodos del capítulo de QAOA
grafo = nx.complete_graph(5)

qubits = [cirq.GridQubit(0, i) for i in range(grafo.number_of_nodes())]

estado_inicial = cirq.Circuit()
for qubit in qubits:
    estado_inicial.append(cirq.H(qubit))  # superposición de todas las divisiones posibles

hamiltoniano_mezcla = construir_hamiltoniano_mezcla(qubits)
hamiltoniano_coste = construir_hamiltoniano_coste(grafo, qubits)

p = 1  # número de repeticiones del ciclo coste-mezcla
gamma, beta = sympy.symbols("gamma beta")
circuito_qaoa = cirq.Circuit()
circuito_qaoa += tfq.util.exponential(operators=[hamiltoniano_coste, hamiltoniano_mezcla],
                                       coefficients=[gamma, beta])

entradas = tfq.convert_to_tensor([estado_inicial])
capa_entrada = tf.keras.layers.Input(shape=(), dtype=tf.dtypes.string)
salida = tfq.layers.PQC(circuito_qaoa, hamiltoniano_coste)(capa_entrada)
modelo_qaoa = tf.keras.models.Model(inputs=capa_entrada, outputs=salida)
modelo_qaoa.compile(loss=tf.keras.losses.MAE, optimizer=tf.keras.optimizers.Adam())

# Entrenamos para que el valor esperado del corte se acerque al máximo teórico (etiqueta ficticia 0)
historial = modelo_qaoa.fit(entradas, np.array([0]), epochs=200)

plt.plot(historial.history["loss"])
plt.title("QAOA sobre un grafo completo de 5 nodos — pérdida durante el entrenamiento")
plt.xlabel("Iteración")
plt.ylabel("Pérdida")
plt.show()

# Muestreamos el circuito entrenado para obtener divisiones concretas
parametros_entrenados = modelo_qaoa.trainable_variables
circuito_con_datos = tfq.layers.AddCircuit()(entradas, append=circuito_qaoa)
muestras = tfq.layers.Sample()(circuito_con_datos, symbol_names=[gamma, beta],
                                symbol_values=parametros_entrenados, repetitions=1000)

cortes_obtenidos = []
for bits in muestras.values:
    grupo_uno = [i for i, bit in enumerate(bits) if bit == 1]
    cortes_obtenidos.append(nx.algorithms.cuts.cut_size(grafo, grupo_uno))

# Corte máximo real por fuerza bruta: en un grafo completo de 5 nodos hay pocos subconjuntos,
# así que probarlos todos sigue siendo rápido
mejor_corte = max(
    nx.algorithms.cuts.cut_size(grafo, list(combo))
    for r in range(grafo.number_of_nodes() + 1)
    for combo in itertools.combinations(grafo.nodes(), r)
)

print("Corte medio obtenido:", np.mean(cortes_obtenidos))
print("Mejor corte posible (fuerza bruta):", mejor_corte)

# En un grafo completo, todos los nodos están conectados entre sí, así que el corte máximo se
# consigue repartiendo los nodos en dos grupos lo más parecidos posible en tamaño (con 5 nodos,
# un grupo de 2 y otro de 3). QAOA con p=1 suele acercarse a ese máximo, pero rara vez lo alcanza
# de forma perfecta, igual que ocurría con el ciclo de 6 nodos del capítulo de QAOA.
```

</details>

## Parte B — Reflexión final: casos de uso reales

Elige una de las cuatro aplicaciones que viste en la clase de teoría: química cuántica (VQE), optimización combinatoria (QAOA), aprendizaje por refuerzo cuántico (QRL) o validación de hardware (supremacía cuántica). Busca un artículo publicado en los últimos dos años que la aplique a un problema nuevo, distinto de los ejemplos de este curso.

Escribe un resumen de tres a cinco frases que explique:

- Qué problema resuelve el artículo
- Qué algoritmo cuántico de los que aprendiste en el curso usa (o una variante)
- Qué limitación menciona el propio artículo sobre el estado actual de la tecnología

**Busca el artículo y escribe tu resumen antes de continuar.**

Escribe aquí tu respuesta:

